# Day 4 — Orchestration, Data Quality & Incremental Load
### Pipeline runner · DQ checks · Watermark ingestion · Upsert · Business report
### Retail Analytics DE Project

---

> **Project Day:** 4 · **Layer:** End-to-end  
> **Input:** All bronze / silver / gold tables  
> **Prerequisite:** Days 1–3 complete

| Section | Topic |
|---------|-------|
| 1 | Setup — import config + Spark |
| 2 | Pipeline Orchestrator (stage runner with timing) |
| 3 | Data Quality Checks (null / dup / referential / range) |
| 4 | Incremental Load (watermark + upsert) |
| 5 | Final Business Report (SQL KPIs) |
| 6 | Project Complete — full table inventory |

---
## 1. Setup — Import config + Spark

In [ ]:
import sys, os, time, uuid, json, re
import xml.etree.ElementTree as ET
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from config.db_config import (
    get_engine, set_spark_env, get_spark,
    new_batch, raw_path,
    spark_read_pg, spark_write_pg, JDBC_URL, DB_USER, DB_PASS
)

engine = get_engine()
print('Engine ready:', engine.url)

In [ ]:
set_spark_env()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType, BooleanType

spark = get_spark('Day4_Orchestration')
print('Spark ready:', spark.version)

In [ ]:
from datetime import date, datetime
from sqlalchemy import text, inspect as sa_inspect

BATCH_ID, INGESTED_AT = new_batch()

def log(msg):
    print(f'[{datetime.now().strftime("%H:%M:%S")}] {msg}')

print('Setup complete. Batch:', BATCH_ID)

---
---
## 2. Pipeline Orchestrator

---

A `run_stage()` wrapper that:
- Logs start time with timestamp
- Times execution with `time.time()`
- Catches all exceptions — pipeline continues even if one stage fails
- Records PASS / FAIL for each stage in a dict

This is the same pattern used in production Airflow DAG tasks and Prefect flows.

In [ ]:
pipeline_results = {}

def run_stage(name, fn):
    """Run fn(), log timing, record PASS/FAIL — never raises."""
    log(f'START  ── {name}')
    t0 = time.time()
    try:
        fn()
        elapsed = round(time.time() - t0, 1)
        log(f'OK     ── {name}  ({elapsed}s)')
        pipeline_results[name] = ('PASS', elapsed)
    except Exception as e:
        elapsed = round(time.time() - t0, 1)
        log(f'FAILED ── {name}  ({elapsed}s)  →  {e}')
        pipeline_results[name] = ('FAIL', elapsed)

print('run_stage() ready.')

In [ ]:
# ── Define Bronze stage functions using pure PySpark JDBC ─────────────────────

def add_bronze_meta(df, source_file):
    return (
        df
        .withColumn('_source_file', F.lit(source_file))
        .withColumn('_ingested_at', F.lit(INGESTED_AT))
        .withColumn('_batch_id',    F.lit(BATCH_ID))
    )

def to_bronze_pg(df, table):
    spark_write_pg(df, 'bronze', table, mode='overwrite')
    log(f'  bronze.{table}: {df.count()} rows')


def stage_bronze_csv():
    for fname, table in [
        ('customers.csv', 'customers'), ('orders.csv', 'orders'),
        ('order_items.csv', 'order_items'), ('products.csv', 'products')
    ]:
        df = spark.read.option('header', 'true').option('inferSchema', 'true').csv(raw_path(fname))
        to_bronze_pg(add_bronze_meta(df, fname), table)


def stage_bronze_json_xml():
    with open(raw_path('inventory.json')) as f:
        inv_data = json.load(f)
    df_inv = spark.createDataFrame(inv_data)
    to_bronze_pg(add_bronze_meta(df_inv, 'inventory.json'), 'inventory')

    with open(raw_path('weather_api_response.json')) as f:
        payload = json.load(f)
    records = [{**r, '_api_source': payload['source']} for r in payload['data']]
    df_w = spark.createDataFrame(records)
    to_bronze_pg(add_bronze_meta(df_w, 'weather_api_response.json'), 'weather_file')

    root = ET.parse(raw_path('store_locations.xml')).getroot()
    xml_rows = [{c.tag: c.text for c in s} for s in root.findall('store')]
    df_xml = spark.createDataFrame(xml_rows)
    to_bronze_pg(add_bronze_meta(df_xml, 'store_locations.xml'), 'store_locations')


def stage_bronze_logs():
    rows = []
    with open(raw_path('webserver_access.log')) as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            parts = line.split('"')
            if len(parts) < 5:
                continue
            left   = parts[0].strip().split()
            middle = parts[2].strip().split()
            req    = parts[1].split()
            rows.append({
                'ip'           : left[0],
                'user'         : left[2] if left[2] != '-' else None,
                'timestamp'    : left[3].lstrip('['),
                'method'       : req[0]  if len(req) > 0 else None,
                'endpoint'     : req[1]  if len(req) > 1 else None,
                'status_code'  : int(middle[0]) if middle else None,
                'response_size': int(middle[1]) if len(middle) > 1 and middle[1].isdigit() else None,
                'referrer'     : parts[3] if parts[3] != '-' else None,
                'user_agent'   : parts[4].strip(),
                'line_num'     : i,
            })
    df = spark.createDataFrame(rows)
    to_bronze_pg(add_bronze_meta(df, 'webserver_access.log'), 'web_logs')


print('Stage functions defined.')

In [ ]:
# ── Run the pipeline ───────────────────────────────────────────────────────────
log('=' * 55)
log('Retail Analytics DE Pipeline — Bronze Run')
log('=' * 55)

run_stage('Bronze: CSV files',  stage_bronze_csv)
run_stage('Bronze: JSON + XML', stage_bronze_json_xml)
run_stage('Bronze: Web Logs',   stage_bronze_logs)

log('=' * 55)
log('Pipeline Summary:')
for name, (status, elapsed) in pipeline_results.items():
    mark = '✓' if status == 'PASS' else '✗'
    log(f'  [{mark}] {status}  {name}  ({elapsed}s)')

In [ ]:
# Row count audit after pipeline run
bronze_tables = ['customers', 'orders', 'order_items', 'products',
                 'inventory', 'store_locations', 'web_logs', 'weather_file']

print(f'\n{"Table":<30} {"Rows":>6}')
print('-' * 38)
for t in bronze_tables:
    try:
        with engine.connect() as conn:
            n = conn.execute(text(f'SELECT COUNT(*) FROM bronze.{t}')).scalar()
        print(f'  bronze.{t:<24} {n:>6}')
    except Exception:
        print(f'  bronze.{t:<24}  (missing)')

---
---
## 3. Data Quality Checks

---

Four DQ check types — all run regardless of failures, results collected into a final report:

| Type | What it catches |
|------|-----------------|
| **Null check** | Required columns must have no nulls |
| **Duplicate check** | Primary key columns must be unique |
| **Referential integrity** | FK values must exist in parent table (PySpark left_anti join) |
| **Value range** | Amounts > 0, quantities ≥ 1, stock ≥ 0 |

In [ ]:
dq_results = []

def check(name, passed, detail=''):
    status = 'PASS' if passed else 'FAIL'
    dq_results.append({'Check': name, 'Status': status, 'Detail': detail})
    mark = '✓' if passed else '✗'
    print(f'  [{mark}] {status}  {name} — {detail}')

print('check() helper ready.')

In [ ]:
# Read silver tables into PySpark DataFrames for DQ via JDBC
df_cust  = spark_read_pg(spark, 'silver', 'customers')
df_ord   = spark_read_pg(spark, 'silver', 'orders')
df_items = spark_read_pg(spark, 'silver', 'order_items')
df_prod  = spark_read_pg(spark, 'silver', 'products')
df_inv   = spark_read_pg(spark, 'silver', 'inventory')

print('Silver tables loaded into Spark for DQ.')

In [ ]:
print('── Null Checks ──────────────────────────────────────────')
for col in ['customer_id', 'email', 'full_name']:
    n = df_cust.filter(F.col(col).isNull()).count()
    check(f'customers.{col} not null', n == 0, f'{n} nulls')

for col in ['order_id', 'customer_id', 'order_date', 'total_amount']:
    n = df_ord.filter(F.col(col).isNull()).count()
    check(f'orders.{col} not null', n == 0, f'{n} nulls')

for col in ['item_id', 'order_id', 'product_id', 'line_total']:
    n = df_items.filter(F.col(col).isNull()).count()
    check(f'order_items.{col} not null', n == 0, f'{n} nulls')

In [ ]:
print('── Duplicate Checks ─────────────────────────────────────')
for df, tbl, pk in [
    (df_cust,  'customers',   'customer_id'),
    (df_ord,   'orders',      'order_id'),
    (df_prod,  'products',    'product_id'),
    (df_items, 'order_items', 'item_id'),
]:
    total    = df.count()
    distinct = df.select(pk).distinct().count()
    dups     = total - distinct
    check(f'{tbl}.{pk} unique', dups == 0, f'{dups} duplicates')

In [ ]:
print('── Referential Integrity ────────────────────────────────')
# PySpark left_anti join: returns rows in left that have NO match in right

orphan_ord = df_ord.join(df_cust.select('customer_id'), on='customer_id', how='left_anti').count()
check('orders.customer_id → customers', orphan_ord == 0, f'{orphan_ord} orphans')

orphan_items_o = df_items.join(df_ord.select('order_id'), on='order_id', how='left_anti').count()
check('order_items.order_id → orders', orphan_items_o == 0, f'{orphan_items_o} orphans')

orphan_items_p = df_items.join(df_prod.select('product_id'), on='product_id', how='left_anti').count()
check('order_items.product_id → products', orphan_items_p == 0, f'{orphan_items_p} orphans')

In [ ]:
print('── Value Range Checks ───────────────────────────────────')
df_ord_t   = df_ord.withColumn('total_amount', F.col('total_amount').cast(DoubleType()))
df_items_t = df_items.withColumn('line_total', F.col('line_total').cast(DoubleType())) \
                      .withColumn('quantity',   F.col('quantity').cast(IntegerType()))
df_inv_t   = df_inv.withColumn('stock_qty', F.col('stock_qty').cast(IntegerType()))

neg_amt = df_ord_t.filter(F.col('total_amount') <= 0).count()
check('orders.total_amount > 0',    neg_amt == 0, f'{neg_amt} non-positive')

neg_qty = df_items_t.filter(F.col('quantity') <= 0).count()
check('order_items.quantity > 0',   neg_qty == 0, f'{neg_qty} non-positive')

neg_lt  = df_items_t.filter(F.col('line_total') < 0).count()
check('order_items.line_total >= 0',neg_lt == 0,  f'{neg_lt} negative')

neg_stk = df_inv_t.filter(F.col('stock_qty') < 0).count()
check('inventory.stock_qty >= 0',   neg_stk == 0, f'{neg_stk} negative')

In [ ]:
passed = sum(1 for r in dq_results if r['Status'] == 'PASS')
failed = sum(1 for r in dq_results if r['Status'] == 'FAIL')

print('\n' + '=' * 62)
print('  DATA QUALITY REPORT')
print('=' * 62)
print(f'  {"Check":<45} {"Status":<6} {"Detail"}')
print('-' * 62)
for r in dq_results:
    mark = '✓' if r['Status'] == 'PASS' else '✗'
    print(f'  [{mark}] {r["Check"]:<43} {r["Status"]:<6} {r["Detail"]}')
print(f'\nResult: {passed} PASS  /  {failed} FAIL  out of {len(dq_results)} checks')

---
---
## 4. Incremental Load

---

**Watermark pattern — how incremental ingestion works:**
```
1. Read MAX(order_date) from bronze.orders  →  watermark
2. Incoming batch arrives (simulated here)
3. Filter batch: keep only rows WHERE order_date > watermark
4. Append filtered rows to bronze.orders   (if_exists='append')
5. Upsert to silver.orders                 (ON CONFLICT DO UPDATE)
```

This ensures **idempotency** — re-running the pipeline on the same data never double-inserts.

In [ ]:
# Step 1: watermark
with engine.connect() as conn:
    wm = conn.execute(text('SELECT MAX(order_date) FROM bronze.orders')).scalar()
print(f'Watermark (MAX order_date in bronze.orders): {wm}')

In [ ]:
# Step 2: incoming batch (simulates today's new orders)
today = str(date.today())

new_orders_data = [
    {'order_id': 'O051', 'customer_id': 'C001', 'order_date': today, 'status': 'pending',
     'payment_method': 'credit_card', 'total_amount': 199.99, 'shipping_city': 'New York'},
    {'order_id': 'O052', 'customer_id': 'C003', 'order_date': today, 'status': 'pending',
     'payment_method': 'paypal', 'total_amount': 55.50, 'shipping_city': 'Chicago'},
    {'order_id': 'O053', 'customer_id': 'C005', 'order_date': today, 'status': 'shipped',
     'payment_method': 'credit_card', 'total_amount': 120.00, 'shipping_city': 'Phoenix'},
]

df_new_orders = spark.createDataFrame(new_orders_data)
print(f'Incoming batch ({df_new_orders.count()} rows):')
df_new_orders.show(truncate=False)

In [ ]:
# Step 3: filter newer than watermark
df_filtered = df_new_orders.filter(F.col('order_date') > str(wm))
print(f'After watermark filter: {df_filtered.count()} rows  (watermark = {wm})')

In [ ]:
# Step 4: append to bronze via JDBC (mode='append' — never overwrites existing rows)
if df_filtered.count() > 0:
    df_meta = (
        df_filtered
        .withColumn('_source_file', F.lit('incremental_batch'))
        .withColumn('_ingested_at', F.lit(INGESTED_AT))
        .withColumn('_batch_id',    F.lit(BATCH_ID))
    )
    spark_write_pg(df_meta, 'bronze', 'orders', mode='append')
    print(f'Appended {df_meta.count()} rows to bronze.orders')

with engine.connect() as conn:
    total_bronze = conn.execute(text('SELECT COUNT(*) FROM bronze.orders')).scalar()
print(f'bronze.orders total: {total_bronze}')

In [ ]:
# Step 5: upsert to silver — ON CONFLICT DO UPDATE (via SQLAlchemy text)
try:
    with engine.connect() as conn:
        conn.execute(text('ALTER TABLE silver.orders ADD PRIMARY KEY (order_id);'))
        conn.commit()
    print('PK added to silver.orders')
except Exception as e:
    print(f'PK note: {e}')   # already exists — fine

upsert_sql = text("""
    INSERT INTO silver.orders
        (order_id, customer_id, order_date, status,
         payment_method, total_amount, shipping_city,
         is_cancelled, order_value_tier, _silver_loaded_at)
    VALUES
        (:order_id, :customer_id, :order_date, :status,
         :payment_method, :total_amount, :shipping_city,
         :is_cancelled, :tier, :loaded_at)
    ON CONFLICT (order_id) DO UPDATE SET
        status            = EXCLUDED.status,
        total_amount      = EXCLUDED.total_amount,
        _silver_loaded_at = EXCLUDED._silver_loaded_at
""")

# Collect the filtered rows as Python dicts to iterate for upsert
rows_to_upsert = df_filtered.collect()

with engine.connect() as conn:
    for row in rows_to_upsert:
        amt  = float(row['total_amount'])
        tier = 'high' if amt >= 200 else ('medium' if amt >= 100 else 'low')
        conn.execute(upsert_sql, {
            'order_id'      : row['order_id'],
            'customer_id'   : row['customer_id'],
            'order_date'    : row['order_date'],
            'status'        : row['status'],
            'payment_method': row['payment_method'],
            'total_amount'  : amt,
            'shipping_city' : row['shipping_city'],
            'is_cancelled'  : False,
            'tier'          : tier,
            'loaded_at'     : datetime.utcnow().isoformat(),
        })
    conn.commit()

with engine.connect() as conn:
    total_silver = conn.execute(text('SELECT COUNT(*) FROM silver.orders')).scalar()
print(f'Upserted {len(rows_to_upsert)} rows — silver.orders total: {total_silver}')

---
---
## 5. Final Business Report

---

In [ ]:
print('── 1. OVERALL REVENUE KPIs ──────────────────────────────')
with engine.connect() as conn:
    row = conn.execute(text("""
        SELECT
            COUNT(DISTINCT order_id)                                              AS total_orders,
            COUNT(DISTINCT order_id) FILTER (WHERE is_cancelled)                 AS cancelled,
            ROUND(100.0*COUNT(*) FILTER (WHERE is_cancelled)/COUNT(*), 1)        AS cancel_rate_pct,
            ROUND(SUM(total_amount) FILTER (WHERE NOT is_cancelled)::NUMERIC, 2) AS gross_revenue,
            ROUND(AVG(total_amount) FILTER (WHERE NOT is_cancelled)::NUMERIC, 2) AS avg_order_value,
            COUNT(DISTINCT customer_id)                                           AS unique_customers
        FROM silver.orders
    """)).fetchone()
for k, v in row._mapping.items():
    print(f'  {k:<25}: {v}')

In [ ]:
print('── 2. TOP 5 PRODUCTS BY REVENUE (RANK window) ───────────')
df_top_prod = spark.read.format('jdbc') \
    .option('url', JDBC_URL) \
    .option('dbtable', """(
        SELECT p.product_name, p.category,
            ROUND(SUM(oi.line_total)::NUMERIC, 2)          AS revenue,
            SUM(oi.quantity)                                AS units_sold,
            RANK() OVER (ORDER BY SUM(oi.line_total) DESC) AS rank
        FROM silver.order_items oi
        JOIN silver.products p ON oi.product_id = p.product_id
        GROUP BY p.product_name, p.category
        ORDER BY revenue DESC LIMIT 5
    ) t""") \
    .option('user', DB_USER) \
    .option('password', DB_PASS) \
    .option('driver', 'org.postgresql.Driver') \
    .load()
df_top_prod.show(truncate=False)

In [ ]:
print('── 3. MONTH-OVER-MONTH GROWTH (LAG window) ──────────────')
df_mom = spark_read_pg(spark, 'gold', 'v_mom_growth')
df_mom.show(truncate=False)

In [ ]:
print('── 4. RFM SEGMENT SUMMARY ───────────────────────────────')
df_rfm = spark_read_pg(spark, 'gold', 'customer_rfm_segments')
df_rfm.groupBy('segment').agg(
    F.count('*').alias('customers'),
    F.round(F.avg('monetary'), 2).alias('avg_spend'),
    F.round(F.avg('recency_days'), 1).alias('avg_recency_days'),
).orderBy(F.col('avg_spend').desc()).show(truncate=False)

In [ ]:
print('── 5. INVENTORY HEALTH ──────────────────────────────────')
with engine.connect() as conn:
    row = conn.execute(text("""
        SELECT COUNT(*) AS total_skus,
            COUNT(*) FILTER (WHERE is_low_stock)  AS low_stock,
            COUNT(*) FILTER (WHERE stock_qty = 0) AS out_of_stock,
            ROUND(100.0*COUNT(*) FILTER (WHERE is_low_stock)/COUNT(*), 1) AS low_stock_pct
        FROM silver.inventory
    """)).fetchone()
for k, v in row._mapping.items():
    print(f'  {k:<20}: {v}')

print('\nLow stock products:')
df_low = spark.read.format('jdbc') \
    .option('url', JDBC_URL) \
    .option('dbtable', """(
        SELECT i.product_id, p.product_name, i.stock_qty,
               i.reorder_level, i.days_since_update
        FROM silver.inventory i
        JOIN silver.products p ON i.product_id = p.product_id
        WHERE i.is_low_stock = TRUE
        ORDER BY i.stock_qty
    ) t""") \
    .option('user', DB_USER) \
    .option('password', DB_PASS) \
    .option('driver', 'org.postgresql.Driver') \
    .load()
df_low.show(truncate=False)

In [ ]:
print('── 6. WEB TRAFFIC SUMMARY ───────────────────────────────')
with engine.connect() as conn:
    row = conn.execute(text("""
        SELECT COUNT(*) AS total_requests, COUNT(DISTINCT ip) AS unique_ips,
            COUNT(*) FILTER (WHERE status_code = 200)  AS ok_200,
            COUNT(*) FILTER (WHERE status_code >= 400) AS errors,
            ROUND(100.0*COUNT(*) FILTER (WHERE status_code>=400)/COUNT(*),1) AS error_rate_pct
        FROM silver.web_logs
    """)).fetchone()
for k, v in row._mapping.items():
    print(f'  {k:<20}: {v}')

---
---
## 6. Project Complete — Full Table Inventory

---

In [ ]:
print('=' * 60)
print('  RETAIL ANALYTICS — MEDALLION PIPELINE COMPLETE')
print('=' * 60)

insp = sa_inspect(engine)
print(f'\n{"Layer":<8} {"Table":<30} {"Rows":>6}')
print('-' * 48)
for schema in ['bronze', 'silver', 'gold']:
    for t in sorted(insp.get_table_names(schema=schema)):
        try:
            with engine.connect() as conn:
                n = conn.execute(text(f'SELECT COUNT(*) FROM {schema}.{t}')).scalar()
            print(f'  {schema:<8} {t:<30} {n:>6}')
        except Exception:
            print(f'  {schema:<8} {t:<30}  (error)')

print('\nAll 4 days complete — Bronze → Silver → Gold pipeline done!')

In [ ]:
spark.stop()
print('SparkSession stopped.')